# Fine-tune YOLO on SoccerNet-v3 (ball / players / goalposts)

Round 2: **continue from your trained `best.pt`**. Do not start from `yolov8m.pt`.

Focus: **small ball**, players, goalposts. More labeled games + larger images.

## Kaggle setup (before Run All)

1. **File → Import notebook** → this file
2. **Settings → Accelerator:** **GPU T4 x2** (not P100)
3. **Settings → Internet:** On
4. **Add-ons → Secrets:** `SOCCERNET_PASSWORD`
5. **REQUIRED — Add Input → Models:** upload `models/best.pt` (the SoccerNet weights you just trained). The notebook **stops** if this is missing.
6. **Run All** (~2–4 hours)

## After training

```bash
cp ~/Downloads/best.pt ~/Desktop/statsapp/models/best.pt
cd ~/Desktop/statsapp
python yolo_inference.py
```

Watch `output_videos/yolo_annotated.mp4` — the ball should stick on more frames.

**Success check:** convert cell prints `goalpost labels: > 0` and `ball labels: > 0`. Train cell must print `FINE-TUNING from: /kaggle/input/.../best.pt`.

In [ ]:
!pip install -q ultralytics SoccerNet tqdm

In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Settings → Accelerator → GPU T4, then Restart & Run All.")

name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
print(f"GPU: {name}  capability: {major}.{minor}")
if major < 7:
    raise SystemExit(
        f"{name} is too old for this PyTorch (needs sm_70+).\n"
        "Kaggle Settings → Accelerator → GPU T4 (NOT P100). Then Restart Session & Run All."
    )
print("GPU OK")


In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["SOCCERNET_PASSWORD"] = UserSecretsClient().get_secret("SOCCERNET_PASSWORD")
    print("Password loaded from Kaggle Secrets")
except Exception as e:
    # Fallback: paste password here only if secrets fail
    # os.environ["SOCCERNET_PASSWORD"] = "YOUR_PASSWORD_HERE"
    raise SystemExit(
        "Add Kaggle Secret SOCCERNET_PASSWORD (Add-ons → Secrets). "
        f"Detail: {e}"
    )

In [ ]:
from pathlib import Path

from SoccerNet.Downloader import SoccerNetDownloader
from SoccerNet.utils import getListGames

SOCCERNET_DIR = Path("/kaggle/working/SoccerNet")
YOLO_DIR = Path("/kaggle/working/soccernet_yolo")
MAX_GAMES = 45  # more labels than round 1 (30). Drop to 35 if disk fills.

SOCCERNET_DIR.mkdir(parents=True, exist_ok=True)

downloader = SoccerNetDownloader(LocalDirectory=str(SOCCERNET_DIR))
downloader.password = os.environ["SOCCERNET_PASSWORD"]

games = getListGames("train", task="frames")[:MAX_GAMES]
print(f"Downloading {len(games)} labeled games (Labels-v3 + Frames-v3)...")

for i, game in enumerate(games, 1):
    print(f"[{i}/{len(games)}] {game}")
    downloader.downloadGame(
        game=game,
        files=["Frames-v3.zip", "Labels-v3.json"],
        spl="train",
    )

!df -h /kaggle/working
print("Download complete")


In [ ]:
from pathlib import Path

Path("/kaggle/working/soccernet_to_yolo.py").write_text(r'''
from __future__ import annotations
import json, shutil, zipfile
from pathlib import Path
from tqdm import tqdm

YOLO_NAMES = ["ball", "player", "goalkeeper", "referee", "goalpost"]
SN_BBOX_CLASS_TO_YOLO = {
    "Ball": 0,
    "Player team left": 1, "Player team right": 1,
    "Player team unknown 1": 1, "Player team unknown 2": 1,
    "Goalkeeper team left": 2, "Goalkeeper team right": 2, "Goalkeeper team unknown": 2,
    "Main referee": 3, "Side referee": 3,
}
SN_GOAL_LINE_CLASS_TO_YOLO = {
    "Goal left post left ": 4, "Goal left post right": 4, "Goal left crossbar": 4,
    "Goal right post left": 4, "Goal right post right": 4, "Goal right crossbar": 4,
}

def _xyxy_to_yolo_line(cls_id, x1, y1, x2, y2, image_meta):
    w_img = float(image_meta["width"]); h_img = float(image_meta["height"])
    x_c = ((x1 + x2) / 2.0) / w_img; y_c = ((y1 + y2) / 2.0) / h_img
    bw = abs(x2 - x1) / w_img; bh = abs(y2 - y1) / h_img
    x_c = min(1.0, max(0.0, x_c)); y_c = min(1.0, max(0.0, y_c))
    bw = min(1.0, max(0.0, bw)); bh = min(1.0, max(0.0, bh))
    if bw <= 0 or bh <= 0: return None
    return f"{cls_id} {x_c:.6f} {y_c:.6f} {bw:.6f} {bh:.6f}"

def bbox_to_yolo_line(bbox, image_meta):
    sn = bbox.get("class")
    if sn not in SN_BBOX_CLASS_TO_YOLO: return None
    p = bbox["points"]
    return _xyxy_to_yolo_line(SN_BBOX_CLASS_TO_YOLO[sn], float(p["x1"]), float(p["y1"]), float(p["x2"]), float(p["y2"]), image_meta)

def line_to_yolo_line(line, image_meta, padding=12.0):
    sn = line.get("class")
    if sn not in SN_GOAL_LINE_CLASS_TO_YOLO: return None
    pts = line.get("points") or []
    if len(pts) < 4: return None
    xs = [float(pts[i]) for i in range(0, len(pts), 2)]
    ys = [float(pts[i]) for i in range(1, len(pts), 2)]
    return _xyxy_to_yolo_line(SN_GOAL_LINE_CLASS_TO_YOLO[sn], min(xs)-padding, min(ys)-padding, max(xs)+padding, max(ys)+padding, image_meta)

def _extract_image(zip_path, image_name, dest_path):
    if dest_path.exists(): return True
    if not zip_path.exists(): return False
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        if image_name not in zf.namelist(): return False
        with zf.open(image_name) as src, open(dest_path, "wb") as dst:
            shutil.copyfileobj(src, dst)
    return True

def convert_game(soccernet_root, game_rel_path, images_out, labels_out, stem_prefix):
    game_dir = soccernet_root / game_rel_path
    labels_path = game_dir / "Labels-v3.json"
    if not labels_path.exists(): return 0
    metadata = json.loads(labels_path.read_text(encoding="utf-8"))
    zip_path = soccernet_root / metadata["GameMetadata"]["UrlLocal"] / "Frames-v3.zip"
    count = 0
    for action_name in metadata["GameMetadata"]["list_actions"]:
        img_names = [action_name] + metadata["actions"][action_name]["linked_replays"]
        for i, img_name in enumerate(img_names):
            img_type = "actions" if i == 0 else "replays"
            ann = metadata[img_type][img_name]
            lines = []
            for bbox in ann.get("bboxes", []):
                line = bbox_to_yolo_line(bbox, ann["imageMetadata"])
                if line: lines.append(line)
            for goal_line in ann.get("lines", []):
                line = line_to_yolo_line(goal_line, ann["imageMetadata"])
                if line: lines.append(line)
            if not lines: continue
            safe_stem = f"{stem_prefix}_{img_name.replace('/', '_').replace('.png', '')}"
            image_out = images_out / f"{safe_stem}.png"
            label_out = labels_out / f"{safe_stem}.txt"
            if not _extract_image(zip_path, img_name, image_out): continue
            label_out.write_text("\n".join(lines) + "\n", encoding="utf-8")
            count += 1
    return count

def write_data_yaml(output_dir):
    yaml_path = output_dir / "data.yaml"
    val = output_dir / "images" / "val"
    val_path = "images/val" if val.exists() and any(val.glob("*")) else "images/train"
    yaml_path.write_text(
        f"path: {output_dir.resolve()}\ntrain: images/train\nval: {val_path}\ntest: {val_path}\nnc: {len(YOLO_NAMES)}\nnames: {YOLO_NAMES}\n",
        encoding="utf-8",
    )
    return yaml_path

def convert_soccernet_v3(soccernet_root, output_dir, splits=None, max_games_per_split=None):
    from SoccerNet.utils import getListGames
    soccernet_root = Path(soccernet_root); output_dir = Path(output_dir)
    splits = splits or ["train"]
    split_map = {"train": "train", "valid": "val", "test": "test"}
    total = 0
    for split in splits:
        yolo_split = split_map.get(split, split)
        images_out = output_dir / "images" / yolo_split
        labels_out = output_dir / "labels" / yolo_split
        images_out.mkdir(parents=True, exist_ok=True)
        labels_out.mkdir(parents=True, exist_ok=True)
        games = getListGames(split, task="frames")
        if max_games_per_split is not None:
            games = games[:max_games_per_split]
        for game in tqdm(games, desc=f"Converting {split}"):
            prefix = game.replace("/", "_").replace(" ", "_")
            total += convert_game(soccernet_root, game, images_out, labels_out, prefix)
    yaml_path = write_data_yaml(output_dir)
    print(f"Converted {total} images -> {output_dir}")
    return yaml_path
''')
print("Wrote /kaggle/working/soccernet_to_yolo.py")

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, "/kaggle/working")
from soccernet_to_yolo import convert_soccernet_v3

yaml_path = convert_soccernet_v3(
    soccernet_root=SOCCERNET_DIR,
    output_dir=YOLO_DIR,
    splits=["train"],
    max_games_per_split=MAX_GAMES,
)
print("data.yaml:", yaml_path)

gp = 0
ball = 0
for p in Path(YOLO_DIR).rglob("*.txt"):
    for line in p.read_text().splitlines():
        if line.startswith("0 "):
            ball += 1
        elif line.startswith("4 "):
            gp += 1

print(f"ball labels: {ball}")
print(f"goalpost labels: {gp}")
assert gp > 0, "No goalpost labels — converter failed; do not train"
assert ball > 0, "No ball labels — check download"
print("OK: labeled SoccerNet export ready")

In [ ]:
from pathlib import Path
from ultralytics import YOLO

candidates = list(Path("/kaggle/input").rglob("best.pt"))
if not candidates:
    raise SystemExit(
        "Upload your trained best.pt: Add Input → Models.\n"
        "This run must FINE-TUNE that model, not start from yolov8m.pt."
    )

MODEL = str(candidates[0])
print("FINE-TUNING from:", MODEL)

model = YOLO(MODEL)
results = model.train(
    data=str(yaml_path),
    epochs=80,
    imgsz=960,       # larger image → better small ball
    batch=8,         # T4 memory at 960
    patience=15,
    project="/kaggle/working/runs",
    name="soccernet_v3_ball",
    device=0,
    lr0=5e-5,        # low LR: continuing, not restarting
    lrf=0.01,
    cls=1.5,         # push classification (ball vs clutter)
    copy_paste=0.4,  # extra copies of rare objects (ball)
    mosaic=1.0,
    close_mosaic=20,
    scale=0.7,
    hsv_s=0.8,
    hsv_v=0.5,
)
BEST = results.save_dir / "weights" / "best.pt"
print("Best weights:", BEST)


In [ ]:
from IPython.display import FileLink
from pathlib import Path
from ultralytics import YOLO

found = list(Path("/kaggle/working/runs").rglob("best.pt"))
assert found, "No best.pt found"
best = found[0]

metrics = YOLO(str(best)).val(data=str(yaml_path))
print(metrics)
print("Download this file:", best)
print("Target: ball mAP50-95 well above the previous 0.49")
FileLink(str(best))
